# 03. Xuất dữ liệu cho Power BI (Star Schema)

Xuất dữ liệu theo mô hình **star schema**, sẵn sàng import thẳng vào
Power BI / Tableau / Looker Studio - không cần xử lý thêm.

**Input:** `outputs/clean_reviews.csv`, `outputs/absa_results.csv`, `outputs/absa_review_summary.csv`

**Output:**
- `outputs/bi_export/dim_brand.csv`
- `outputs/bi_export/dim_product.csv`
- `outputs/bi_export/dim_date.csv`
- `outputs/bi_export/fact_reviews.csv` (grain = 1 review)
- `outputs/bi_export/fact_absa.csv` (grain = 1 review x 1 aspect - dùng để phân tích/lọc theo khía cạnh)

> **Lưu ý quan trọng** (đã kiểm tra khi xây notebook 01): `price_current`,
> `price_original`, `sold_count` là thông tin **cấp sản phẩm** (lặp lại y
> hệt trên mọi review cùng `product_id`) → đã được dedup sẵn trong
> `dim_product`, không cần xử lý lại trong Power BI.

Hướng dẫn dựng dashboard chi tiết (relationships, DAX measures, layout)
xem tại `../POWERBI_GUIDE.md`.

In [1]:
import pandas as pd

OUT_DIR = "../outputs"
BI_DIR = f"{OUT_DIR}/bi_export"

df = pd.read_csv(f"{OUT_DIR}/clean_reviews.csv")
absa = pd.read_csv(f"{OUT_DIR}/absa_results.csv")
review_summary = pd.read_csv(f"{OUT_DIR}/absa_review_summary.csv")

## dim_brand

In [2]:
dim_brand = df[["brand_name"]].drop_duplicates().reset_index(drop=True)
dim_brand.insert(0, "brand_id", range(1, len(dim_brand) + 1))
dim_brand.to_csv(f"{BI_DIR}/dim_brand.csv", index=False)
dim_brand

,brand_id,brand_name
0,1,Murad
1,2,Obagi
2,3,URIAGE
3,4,Eucerin
4,5,CeraVe


## dim_product (đã dedup theo product_id)

In [3]:
dim_product = df[
    ["product_id", "product_name", "brand_name", "price_current", "price_original", "sold_count"]
].drop_duplicates(subset="product_id").reset_index(drop=True)
dim_product["discount_pct"] = (
    (dim_product["price_original"] - dim_product["price_current"]) / dim_product["price_original"] * 100
).round(1)
dim_product.to_csv(f"{BI_DIR}/dim_product.csv", index=False)
dim_product.head()

,product_id,product_name,brand_name,price_current,price_original,sold_count,discount_pct
0,2070262528,[Shock Deal 02.02 | 500K Off] Essence That Fad...,Murad,2372000,2445000.0,1800,3.0
1,1999474295,[Mega Tet | Discount 500K] Murad Rapid Relie...,Murad,753000,776000.0,2100,3.0
2,1999468969,Murad Nutrient-Charged Water Gel Rich in Essen...,Murad,1998000,1998000.0,139,0.0
3,1999570180,"[Pre-Order] Murad Vita-C Glycolic Serum 30ml, ...",Murad,2472000,2548000.0,622,3.0
4,1999512677,Retinol Moisturizing Cream Helps Smooth Skin a...,Murad,2812000,2812000.0,114,0.0


## dim_date

In [4]:
dim_date = df[["review_year", "review_month"]].drop_duplicates().reset_index(drop=True)
dim_date["date_key"] = pd.to_datetime(
    dim_date["review_year"].astype(int).astype(str) + "-" +
    dim_date["review_month"].astype(int).astype(str) + "-01"
)
dim_date = dim_date.sort_values("date_key").reset_index(drop=True)
dim_date.to_csv(f"{BI_DIR}/dim_date.csv", index=False)
dim_date.head()

,review_year,review_month,date_key
0,2019.0,11.0,2019-11-01
1,2019.0,12.0,2019-12-01
2,2020.0,6.0,2020-06-01
3,2020.0,7.0,2020-07-01
4,2020.0,8.0,2020-08-01


## fact_reviews (grain = 1 review)

In [5]:
fact = df.merge(
    review_summary[["review_id", "primary_sentiment", "has_negative_aspect",
                     "has_positive_aspect", "is_mixed_sentiment", "hidden_complaint"]],
    on="review_id", how="left"
)
fact["date_key"] = pd.to_datetime(
    fact["review_year"].astype(int).astype(str) + "-" +
    fact["review_month"].astype(int).astype(str) + "-01"
)
aspect_cols = [c for c in fact.columns if c.startswith("aspect_")]
keep_cols = (
    ["review_id", "product_id", "brand_name", "date_key", "review_score",
     "review_len", "word_count", "dup_count", "is_template_like",
     "primary_sentiment", "has_negative_aspect", "has_positive_aspect",
     "is_mixed_sentiment", "hidden_complaint"]
    + aspect_cols
)
fact_reviews = fact[keep_cols]
fact_reviews.to_csv(f"{BI_DIR}/fact_reviews.csv", index=False)
fact_reviews.head()

,review_id,product_id,brand_name,date_key,review_score,review_len,word_count,dup_count,is_template_like,primary_sentiment,...,has_positive_aspect,is_mixed_sentiment,hidden_complaint,aspect_Hiệu_quả,aspect_Giá_cả,aspect_Chất_lượng,aspect_Giao_hàng,aspect_Mùi_hương,aspect_Thành_phần,aspect_Dịch_vụ
0,0,2070262528,Murad,2023-06-01,2.0,568,140,1,False,positive,...,True,True,False,0,0,1,0,0,0,1
1,1,2070262528,Murad,2026-04-01,5.0,182,39,1,False,positive,...,True,True,False,1,1,0,0,0,0,0
2,2,2070262528,Murad,2026-04-01,5.0,124,29,1,False,positive,...,True,True,False,1,0,0,0,0,0,0
3,3,2070262528,Murad,2026-03-01,5.0,163,38,1,False,positive,...,True,True,False,1,0,0,0,1,0,0
4,4,2070262528,Murad,2026-03-01,5.0,3,1,3,True,NaN,...,NaN,NaN,NaN,0,0,0,0,0,0,0


## fact_absa (grain = 1 review x 1 aspect, để lọc/slice theo khía cạnh)

In [6]:
fact_absa = absa.merge(
    df[["review_id", "review_year", "review_month"]], on="review_id", how="left"
)
fact_absa["date_key"] = pd.to_datetime(
    fact_absa["review_year"].astype(int).astype(str) + "-" +
    fact_absa["review_month"].astype(int).astype(str) + "-01"
)
fact_absa = fact_absa[
    ["review_id", "product_id", "brand_name", "date_key", "aspect", "clause_sentiment"]
]
fact_absa.to_csv(f"{BI_DIR}/fact_absa.csv", index=False)
fact_absa.head()

,review_id,product_id,brand_name,date_key,aspect,clause_sentiment
0,0,2070262528,Murad,2023-06-01,Chất lượng/Đóng gói,positive
1,0,2070262528,Murad,2023-06-01,Chất lượng/Đóng gói,neutral
2,0,2070262528,Murad,2023-06-01,Chất lượng/Đóng gói,neutral
3,0,2070262528,Murad,2023-06-01,Chất lượng/Đóng gói,neutral
4,0,2070262528,Murad,2023-06-01,Dịch vụ/Uy tín,positive


## Tổng kết

In [7]:
print("Đã xuất star schema vào outputs/bi_export/:")
print(f"  dim_brand:     {dim_brand.shape}")
print(f"  dim_product:   {dim_product.shape}")
print(f"  dim_date:      {dim_date.shape}")
print(f"  fact_reviews:  {fact_reviews.shape}")
print(f"  fact_absa:     {fact_absa.shape}")

Đã xuất star schema vào outputs/bi_export/:
  dim_brand:     (5, 2)
  dim_product:   (152, 7)
  dim_date:      (72, 3)
  fact_reviews:  (7364, 21)
  fact_absa:     (12200, 6)
